# Verify the published numbers

Patil, Kumbhar & Jones, "Contributions of Extended-Range Electric Vehicles (EREVs) to
Electrified Miles, Emissions and Transportation Cost Reduction", *Energies* **2025**, 18,
6448, <https://doi.org/10.3390/en18246448>.

This notebook checks the article's printed tables, and the numbers quoted in its prose,
against this repository's implementation. It is committed with its outputs, so you can read the
result without running anything.

**What it needs:** this repository and `pip install -e .`. **What it does not need:** a
network connection, a Census API key, Excel, or the frozen notebooks. It reads only the
frozen inputs in `data/raw/` and the printed tables in `tests/fixtures/article/`.
`tests/test_verification_notebook.py` re-runs it with the network blocked and no API key
set, so that claim is tested rather than asserted.

The same comparisons run in the test suite (`erev_vmtalloc.verify`), so this notebook and
`pytest` cannot disagree about what "reproduces" means.

## 1. Find the repository and load the configuration

In [1]:
from pathlib import Path

import erev_vmtalloc
from erev_vmtalloc import costs, sources, verify
from erev_vmtalloc.allocation import fit_bin_distances
from erev_vmtalloc.config import load_config

# The notebook may be opened from notebooks/ or from the repository root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "base.yaml").exists())
RAW = ROOT / "data" / "raw"
ARTICLE = ROOT / "tests" / "fixtures" / "article"
config = load_config(ROOT / "config" / "base.yaml")
print("erev_vmtalloc", erev_vmtalloc.__version__)
print("inputs:", sorted(p.name for p in RAW.glob("*.csv")) + sorted(p.name for p in RAW.glob("*.json")))

erev_vmtalloc 1.0.0
inputs: ['charging_gas_vmt_2023.csv', 'fhwa_vmt_2023.csv', 'bts_trips_2023.json']


## 2. Fit the bin distances from the frozen 2023 snapshots

BTS trip counts by distance bin, fitted to FHWA monthly VMT. This is the step every table
depends on.

In [2]:
fitted = fit_bin_distances(
    sources.load_bts_trips(RAW, config.year),
    sources.load_fhwa_vmt(RAW, config.year),
    config["bin_bounds"],
    normalize_to_fhwa=config.normalize_to_fhwa,
)
print(f"annual VMT, normalised to FHWA: {fitted.annual_vmt_billion:,.1f} B miles")
print(f"sum of Table 1's printed VMT column: {costs.table1_printed_total_b(fitted):,.3f} B miles")

annual VMT, normalised to FHWA: 3,262.8 B miles
sum of Table 1's printed VMT column: 3,392.497 B miles


## 3. Every printed cell of Tables 1â€“3 and 5â€“6

A printed value with *d* decimals is reproduced when the package's number rounds to it.
Nineteen cells in Tables 2 and 3 were truncated in print rather than rounded (January's
modelled VMT is 278.367 B and is printed as 278.36). Those cells are listed by name in
`verify.TRUNCATED`, and each must truncate and must not round. Table 4 holds the inputs, not
results; Section 5 covers it.

In [3]:
results = verify.check_all(fitted, config, ARTICLE)
for result in results:
    print(result)
    for miss in result.misses:
        print("   ", miss)

table1: 30 cells round to print, 0 truncate to print, 0 printed N/A or $-, 0 do not reproduce
table2: 29 cells round to print, 7 truncate to print, 0 printed N/A or $-, 0 do not reproduce
table3: 68 cells round to print, 12 truncate to print, 0 printed N/A or $-, 0 do not reproduce
table5: 240 cells round to print, 0 truncate to print, 0 printed N/A or $-, 0 do not reproduce
table6: 249 cells round to print, 0 truncate to print, 15 printed N/A or $-, 0 do not reproduce


Here is one block side by side: the Average scenario of Table 5.

In [4]:
import pandas as pd

printed = verify.printed_table(ARTICLE, "table5")
ours = costs.scenario_table(fitted, config)
cols = ["Electric VMT (B)", "Total Emissions (Mt CO2)", "CO2 Saved (Mt CO2)"]
side = pd.concat(
    [printed.loc[printed.Scenario == "Average", ["Vehicle Type", *cols]].set_index("Vehicle Type")
            .add_prefix("printed: "),
     ours.loc[ours.Scenario == "Average", ["Vehicle Type", *cols]].set_index("Vehicle Type")
         .round(2).add_prefix("package: ")],
    axis=1,
)
side

,printed: Electric VMT (B),printed: Total Emissions (Mt CO2),printed: CO2 Saved (Mt CO2),package: Electric VMT (B),package: Total Emissions (Mt CO2),package: CO2 Saved (Mt CO2)
Vehicle Type,,,,,,
ICE,0.0,1098.5,0.0,0.00,1098.48,0.00
LDV (25-mile EV range),1936.7,633.7,464.8,1936.75,633.66,464.82
LDV (50-mile EV range),2391.1,524.6,573.9,2391.07,524.62,573.86
LDV (75-mile EV range),2551.0,486.2,612.2,2551.01,486.23,612.24
LDV (100-mile EV range),2710.9,447.8,650.6,2710.95,447.85,650.63
LDV (125-mile EV range),2771.6,433.3,665.2,2771.58,433.30,665.18
LDV (150-mile EV range),2832.2,418.7,679.7,2832.20,418.75,679.73
EV,3262.8,315.4,783.1,3262.80,315.40,783.07


## 4. Numbers quoted in the prose

In [5]:
prose_misses = []
for section, quote, value, printed_text in verify.prose_numbers(fitted, config):
    target, decimals = verify.parse(printed_text)
    ok = verify.rounds_to(float(value), target, decimals)
    prose_misses += [] if ok else [quote]
    print(f"{'ok  ' if ok else 'MISS'} {section:<11} {quote:<42} package {float(value):.6g}")

ok   summary     73.3% of VMT                               package 73.2827
ok   summary     2.391 T electric mi/yr                     package 2.39107
ok   summary     86.8% at 150 miles                         package 86.8028
ok   summary     (2.83 T)                                   package 2.8322
ok   summary     574 Mt (50 mi)                             package 573.856
ok   summary     0.072 USD/kg CO2 (50 mile)                 package 0.0717374
ok   4.3         2.39 trillion electric miles               package 2.39107
ok   4.3         573.9 Mt CO2 avoided                       package 573.856
ok   4.3         Battery capacity per VMT 1.097 (50 mi)     package 1.09714
ok   4.3         to 3.291 (150 mi)                          package 3.29141
ok   4.3         +320 B from 50 -> 100                      package 319.879
ok   4.3         +121 B from 100 -> 150                     package 121.256
ok   4.3         battery capacity from 3.6 [50 mi]          package 3.57974
ok   4.3   

## 5. Where the article disagrees with itself

None of these is corrected; each is pinned in `tests/test_paper_numbers.py` and described
in the README under "What reproduces, and what doesn't". The largest is about the VMT total
behind the charging-frequency figures (5, 7, 9, 11, 13):

In [6]:
chg = costs.charging_table(fitted, config, RAW)
seven = chg[chg["Charges / week"] == 7].set_index("Vehicle Type")
avg = ours[ours.Scenario == "Average"].set_index("Vehicle Type")
gap = seven["Electric VMT (B)"] / avg.loc[seven.index, "Electric VMT (B)"] - 1
print(f"charging figures stand on {seven['VMT (B)'].iloc[0]:,.3f} B miles; "
      f"the tables on {fitted.annual_vmt_billion:,.1f} B")
print("7-a-week electric VMT above the Average scenario, by range:")
print((100 * gap).round(2).to_string())

charging figures stand on 3,392.497 B miles; the tables on 3,262.8 B
7-a-week electric VMT above the Average scenario, by range:
Vehicle Type
LDV (25-mile EV range)     3.98
LDV (50-mile EV range)     3.98
LDV (75-mile EV range)     3.98
LDV (100-mile EV range)    3.98
LDV (125-mile EV range)    3.98
LDV (150-mile EV range)    3.98


## 6. Verdict

In [7]:
cells = sum(r.rounded + r.truncated for r in results)
assert all(r.ok for r in results), [m for r in results for m in r.misses]
assert not prose_misses, prose_misses
print(f"VERDICT: {cells} printed table cells and "
      f"{len(verify.prose_numbers(fitted, config))} prose numbers reproduce; "
      f"{sum(len(r.misses) for r in results)} do not.")

VERDICT: 635 printed table cells and 31 prose numbers reproduce; 0 do not.
